# SIP POWER: read, start, stop

Use the repository `.venv` as the notebook kernel (`uv sync --group notebook`).
Run cells individually: **Start enables real pump HV; Stop disables it.**
The controller itself stays powered. No InfluxDB access.

Keepalive is left unchanged. After remote Start, keep polling with this client
before its keepalive interval expires, or the controller may stop HV and raise
a communication alarm. Pausing between cells can trigger this watchdog.


In [ ]:
import sys
import time
import tomllib
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "py-seas-sip-power":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from saes_sip_power_client import SAESSIPPowerClient, SAESSIPPowerSettings  # noqa: E402

settings = tomllib.loads((ROOT / "settings.toml").read_text())
client = SAESSIPPowerClient(SAESSIPPowerSettings(
    host=settings["host"],
    port=settings.get("port", 2527),
    timeout_s=settings["timeout_s"],
))
client.connect()


## Read current state
Inspect `enabled`, alarms, voltage and `keepalive_interval_ms` before Start.

In [ ]:
status = client.read_sample()
status


## Start HV and watch 10 samples
This sends Start once. Readback shows the observed state, not an ACK.
An interlock or alarm may prevent output; inspect the readings.
After this cell ends, run Stop or continue polling before keepalive expires.


In [ ]:
poll_s = min(1.0, status.keepalive_interval_ms / 2000) if status.keepalive_interval_ms else 1.0
client.start()
for _ in range(10):
    status = client.read_sample()
    print(status.observed_at, "enabled:", status.enabled,
          "V:", status.output_voltage_v, "nA:", status.output_current_na,
          "alarm:", status.global_alarm)
    time.sleep(poll_s)


## Stop HV and read back
Re-run the read cell if the output has not settled yet.

In [ ]:
client.stop()
status = client.read_sample()
status


## Close connection
Closing the socket does not send Stop; use the Stop cell first.

In [ ]:
client.close()
